# 🧹 Pandas III + NumPy — Limpeza e Transformação de Dados

---

## 🎯 Objetivos da Aula

Ao final desta aula, o aluno será capaz de:

1. **Identificar e diagnosticar problemas** em dados reais (valores faltantes, duplicatas, tipos incorretos) usando funções como `isna()`, `isnull()`, `duplicated()` e `info()`
2. **Remover dados problemáticos** com segurança, escolhendo entre `drop()`, `dropna()` e `drop_duplicates()` conforme o contexto
3. **Tratar valores faltantes** (NaN/None) usando diferentes estratégias: valores fixos, média, mediana, forward/backward fill
4. **Transformar e enriquecer colunas** usando `np.where()` e funções customizadas
5. **Aplicar filtros, substituições e acessos** precisos aos dados usando `loc[]`, `iloc[]`, `at[]` e `replace()`

---

## 🔥 Aquecimento — Problema Gerador
Imagine que você trabalha em uma startup de e-commerce. O time de vendas acabou de entregar para você um arquivo Excel com os dados de todos os clientes cadastrados no último trimestre: **5.000 linhas com nomes, e-mails, telefones, data de compra e valor gasto**.

Você abre o arquivo na primeira reunião com o CEO e descobre:
- 🚨 **Coluna "telefone" tem 1.200 valores em branco** — alguns clientes não forneceram
- 🚨 **Coluna "valor_gasto" tem "R$ 150" em texto**, não número — impossível fazer contas
- 🚨 **500 linhas estão duplicadas** — o mesmo cliente foi cadastrado 2 vezes por erro do sistema
- 🚨 **A coluna "data_compra" tem 300 valores faltantes** — o sistema antigo não registrou

O CEO olha para você e pergunta: *"Quanto nossos clientes gastaram realmente? Quantos clientes únicos temos?"*

Você não consegue responder porque os dados **não estão limpos**. Nesta aula, você aprenderá a:
- ✅ **Enxergar o problema** antes que ele vire lixo em uma análise
- ✅ **Limpar e preparar** os dados para análise real
- ✅ **Transformar valores** para o formato que você precisa
- ✅ **Tomar decisões conscientes** sobre o que manter, descartar ou corrigir

**Essa é a realidade de 80% do trabalho de um Data Scientist: não é modelagem com IA, é limpeza de dados.**

---

## 📖 Bloco Teórico + Analogias

### 🔍 Entendendo o Problema: Por Que Dados Precisam de Limpeza?

Pense em um **balcão de atendimento de um banco**. Os clientes chegam e preenchem fichas de cadastro:
- Alguns escrevem de caneta azul, outros de vermelha ❌
- Alguns colocam o CPF, outros deixam em branco ❌
- Alguns escrevem "São Paulo" e outros "SP" ❌
- Alguns repetem a mesma informação duas vezes ❌

O gerente do banco **não pode somar dinheiro** enquanto essas fichas estiverem bagunçadas. Ele precisa:
1. **Verificar** quais fichas estão incompletas ou erradas
2. **Descartar** duplicatas
3. **Padronizar** (ex: sempre "SP", nunca "São Paulo")
4. **Preencher buracos** (ex: tentar ligar para quem deixou telefone vazio)

**Dados sujos = fichas bagunçadas. Python + Pandas = ferramentas para organizar essas fichas.**

### 1️⃣ Identificação de Problemas: Diagnóstico Rápido


#### 📊 **Visão Geral com `shape`, `info()` e `describe()`**

A primeira coisa que você faz ao receber um dataset novo é **"radiografar"** ele — entender quantas linhas, quantas colunas, quais tipos de dados, e onde estão os problemas.

**Analogia:** como um médico que faz um raio-X antes de operar. Você precisa ver a estrutura antes de mexer.

In [2]:
import pandas as pd
import numpy as np

clientes = pd.read_csv("C:/Temp/clientes.csv", encoding='utf-8')
display(clientes)

,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado
0,1619,Cliente_1618,cliente1618@mail.com,5.511996e+12,2024-01-20,149.26,MG
1,4217,Cliente_4216,cliente4216@mail.com,5.511996e+12,2024-03-01,198.54,SP
2,1965,Cliente_1964,cliente1964@mail.com,5.511917e+12,2024-01-03,413.4,RS
3,3830,Cliente_3829,cliente3829@mail.com,5.511999e+12,2024-01-15,261.47,SP
4,2101,Cliente_2100,cliente2100@mail.com,5.511974e+12,2024-01-30,461.09,SP
...,...,...,...,...,...,...,...
5518,3773,Cliente_3772,cliente3772@mail.com,5.511916e+12,2024-03-23,130.71,BA
5519,3795,Cliente_3794,cliente3794@mail.com,5.511932e+12,2024-01-28,414.1,BA
5520,2690,Cliente_2689,cliente2689@mail.com,5.511941e+12,2024-02-18,19.94,RJ
5521,1579,Cliente_1578,cliente1578@mail.com,5.511998e+12,2024-02-28,386.22,BA


In [3]:
# 1. shape -- mostra o número de linhas e colunas
print("Shape da tabela:", clientes.shape)

Shape da tabela: (5523, 7)


In [4]:
# 2. info() — tipo de cada coluna e quantos valores não vazios tem
clientes.info()

<class 'pandas.DataFrame'>
RangeIndex: 5523 entries, 0 to 5522
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cliente_id   5523 non-null   int64  
 1   nome         5523 non-null   str    
 2   email        5349 non-null   str    
 3   telefone     4177 non-null   float64
 4   data_compra  5178 non-null   str    
 5   valor_gasto  5523 non-null   str    
 6   estado       5523 non-null   str    
dtypes: float64(1), int64(1), str(5)
memory usage: 567.7 KB


**Leitura:** Note que `telefone` tem apenas 4.177 valores não-nulos de 5.523 = **1.346 vazios!**

In [5]:
# 3. describe() — estatísticas básicas (funciona bem com números)
clientes.describe()

,cliente_id,telefone
count,5523.000000,4.177000e+03
mean,2486.617418,5.511950e+12
std,1445.266426,2.886485e+07
min,1.000000,5.511900e+12
25%,1233.500000,5.511925e+12
50%,2484.000000,5.511949e+12
75%,3735.500000,5.511976e+12
max,5000.000000,5.512000e+12


---

#### 🔎 **Valores Faltantes: `isna()`, `isnull()`, `notnull()`**

Python chama valores vazios de **NaN** (Not a Number) ou **None**. Você precisa localizá-los antes de trabalhar.


In [6]:
# isna() — retorna True onde há valor faltante
print(clientes['telefone'].isna())

0       False
1       False
2       False
3       False
4       False
        ...  
5518    False
5519    False
5520    False
5521    False
5522     True
Name: telefone, Length: 5523, dtype: bool


In [7]:
# Quantos valores faltam em cada coluna?
print("\nValores faltantes por coluna:")
print(clientes.isna().sum())



Valores faltantes por coluna:
cliente_id        0
nome              0
email           174
telefone       1346
data_compra     345
valor_gasto       0
estado            0
dtype: int64


**Tradução:** 1.346 telefones faltam, 174 e-mails faltam, etc.

In [8]:
# isnull() — exatamente igual a isna()
# (são sinônimos — ambos funcionam)
print(clientes.isnull().sum())


# notnull() — inverso de isna()
print(clientes.notnull().sum())


cliente_id        0
nome              0
email           174
telefone       1346
data_compra     345
valor_gasto       0
estado            0
dtype: int64
cliente_id     5523
nome           5523
email          5349
telefone       4177
data_compra    5178
valor_gasto    5523
estado         5523
dtype: int64


In [9]:
# Qual é a percentagem de dados faltantes por coluna?
print((clientes.isna().sum() / len(clientes))*100)



cliente_id      0.000000
nome            0.000000
email           3.150462
telefone       24.370813
data_compra     6.246605
valor_gasto     0.000000
estado          0.000000
dtype: float64


In [10]:
print((clientes.isna().mean())*100)


cliente_id      0.000000
nome            0.000000
email           3.150462
telefone       24.370813
data_compra     6.246605
valor_gasto     0.000000
estado          0.000000
dtype: float64



**Insight:** 24% dos telefones estão faltando — é um problema!

---
#### 📋 **Duplicatas: `duplicated()` e `drop_duplicates()`**

Duplicatas são **linhas repetidas** — o mesmo cliente cadastrado 2 vezes.

**Analogia:** como duas convites chegando para o mesmo convidado. Você quer contar pessoas, não convites.

In [11]:
# Quantas linhas são duplicadas (comparando TODAS as colunas)?
print(clientes.duplicated().sum())


523


In [12]:
# Quais são essas linhas duplicadas?
display(clientes[clientes.duplicated()])

,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado
206,1056,Cliente_1055,cliente1055@mail.com,NaN,2024-01-08,R$ 164.69,RJ
210,298,Cliente_297,cliente297@mail.com,NaN,NaN,R$ 152.44,SP
278,2032,Cliente_2031,cliente2031@mail.com,5.511977e+12,2024-03-14,343.4,RJ
325,30,Cliente_29,NaN,NaN,NaN,R$ 488.22,RS
404,3790,Cliente_3789,cliente3789@mail.com,5.511938e+12,2024-02-22,244.67,RJ
...,...,...,...,...,...,...,...
5505,3626,Cliente_3625,cliente3625@mail.com,5.511918e+12,2024-01-27,99.25,SP
5515,486,Cliente_485,cliente485@mail.com,NaN,2024-02-28,R$ 32.49,RJ
5519,3795,Cliente_3794,cliente3794@mail.com,5.511932e+12,2024-01-28,414.1,BA
5520,2690,Cliente_2689,cliente2689@mail.com,5.511941e+12,2024-02-18,19.94,RJ


In [13]:
# Às vezes você quer saber duplicatas só de UMA coluna
# Ex: quantos clientes únicos temos? (não contar o mesmo e-mail 2 vezes)

print(clientes.duplicated(subset=['email']).sum())


672


In [14]:
# Marca duplicatas de e-mail no DataFrame

# keep=False marca todas as linhas que aparecem mais de uma vez

display(clientes[clientes.duplicated(subset=['email'], keep=False)])




,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado
0,1619,Cliente_1618,cliente1618@mail.com,5.511996e+12,2024-01-20,149.26,MG
3,3830,Cliente_3829,cliente3829@mail.com,5.511999e+12,2024-01-15,261.47,SP
6,4583,Cliente_4582,cliente4582@mail.com,5.511902e+12,2024-02-25,280.67,RS
19,2844,Cliente_2843,cliente2843@mail.com,5.511919e+12,2024-03-08,122.74,MG
20,3050,Cliente_3049,cliente3049@mail.com,5.511994e+12,2024-03-29,334.43,MG
...,...,...,...,...,...,...,...
5510,131,Cliente_130,NaN,NaN,NaN,R$ 118.93,BA
5515,486,Cliente_485,cliente485@mail.com,NaN,2024-02-28,R$ 32.49,RJ
5519,3795,Cliente_3794,cliente3794@mail.com,5.511932e+12,2024-01-28,414.1,BA
5520,2690,Cliente_2689,cliente2689@mail.com,5.511941e+12,2024-02-18,19.94,RJ


In [15]:
# keep='first' marca como duplicada a partir da segunda ocorrência
# keep='last' marca como duplicada a partir da primeira ocorrência

clientes['duplicado_email_first'] = clientes.duplicated(subset=['email'], keep='first')
clientes['duplicado_email_last'] = clientes.duplicated(subset=['email'], keep='last')

display(clientes[clientes['duplicado_email_first'] == True].sort_values(by='email'))

,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado,duplicado_email_first,duplicado_email_last
3542,1002,Cliente_1001,cliente1001@mail.com,NaN,2024-01-06,R$ 264.66,RJ,True,False
5099,1021,Cliente_1020,cliente1020@mail.com,NaN,2024-01-19,R$ 174.29,RS,True,False
1582,1030,Cliente_1029,cliente1029@mail.com,NaN,2024-01-28,R$ 274.62,BA,True,False
3436,1053,Cliente_1052,cliente1052@mail.com,NaN,2024-02-09,R$ 343.24,RS,True,False
206,1056,Cliente_1055,cliente1055@mail.com,NaN,2024-01-08,R$ 164.69,RJ,True,False
...,...,...,...,...,...,...,...,...,...
5373,94,Cliente_93,NaN,NaN,NaN,R$ 375.08,RJ,True,True
5430,65,Cliente_64,NaN,NaN,NaN,R$ 146.73,RJ,True,True
5464,35,Cliente_34,NaN,NaN,NaN,R$ 408.22,RS,True,True
5478,70,Cliente_69,NaN,NaN,NaN,R$ 403.19,BA,True,True


In [16]:
display(clientes)

,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado,duplicado_email_first,duplicado_email_last
0,1619,Cliente_1618,cliente1618@mail.com,5.511996e+12,2024-01-20,149.26,MG,False,True
1,4217,Cliente_4216,cliente4216@mail.com,5.511996e+12,2024-03-01,198.54,SP,False,False
2,1965,Cliente_1964,cliente1964@mail.com,5.511917e+12,2024-01-03,413.4,RS,False,False
3,3830,Cliente_3829,cliente3829@mail.com,5.511999e+12,2024-01-15,261.47,SP,False,True
4,2101,Cliente_2100,cliente2100@mail.com,5.511974e+12,2024-01-30,461.09,SP,False,False
...,...,...,...,...,...,...,...,...,...
5518,3773,Cliente_3772,cliente3772@mail.com,5.511916e+12,2024-03-23,130.71,BA,False,False
5519,3795,Cliente_3794,cliente3794@mail.com,5.511932e+12,2024-01-28,414.1,BA,True,False
5520,2690,Cliente_2689,cliente2689@mail.com,5.511941e+12,2024-02-18,19.94,RJ,True,False
5521,1579,Cliente_1578,cliente1578@mail.com,5.511998e+12,2024-02-28,386.22,BA,True,False


#### ✂️ **Remover Colunas com `drop()`**


In [17]:

# Remove UMA coluna
clientes2 = clientes.drop(columns=['duplicado_email_first'])
#ou
clientes2 = clientes.drop('duplicado_email_first', axis=1)
display(clientes2)




,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado,duplicado_email_last
0,1619,Cliente_1618,cliente1618@mail.com,5.511996e+12,2024-01-20,149.26,MG,True
1,4217,Cliente_4216,cliente4216@mail.com,5.511996e+12,2024-03-01,198.54,SP,False
2,1965,Cliente_1964,cliente1964@mail.com,5.511917e+12,2024-01-03,413.4,RS,False
3,3830,Cliente_3829,cliente3829@mail.com,5.511999e+12,2024-01-15,261.47,SP,True
4,2101,Cliente_2100,cliente2100@mail.com,5.511974e+12,2024-01-30,461.09,SP,False
...,...,...,...,...,...,...,...,...
5518,3773,Cliente_3772,cliente3772@mail.com,5.511916e+12,2024-03-23,130.71,BA,False
5519,3795,Cliente_3794,cliente3794@mail.com,5.511932e+12,2024-01-28,414.1,BA,False
5520,2690,Cliente_2689,cliente2689@mail.com,5.511941e+12,2024-02-18,19.94,RJ,False
5521,1579,Cliente_1578,cliente1578@mail.com,5.511998e+12,2024-02-28,386.22,BA,False


In [18]:
# Remove VÁRIAS colunas
clientes2 = clientes.copy()
display(clientes2)
clientes2 = clientes.drop(columns=['duplicado_email_first', 'duplicado_email_last'])
display(clientes2)




,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado,duplicado_email_first,duplicado_email_last
0,1619,Cliente_1618,cliente1618@mail.com,5.511996e+12,2024-01-20,149.26,MG,False,True
1,4217,Cliente_4216,cliente4216@mail.com,5.511996e+12,2024-03-01,198.54,SP,False,False
2,1965,Cliente_1964,cliente1964@mail.com,5.511917e+12,2024-01-03,413.4,RS,False,False
3,3830,Cliente_3829,cliente3829@mail.com,5.511999e+12,2024-01-15,261.47,SP,False,True
4,2101,Cliente_2100,cliente2100@mail.com,5.511974e+12,2024-01-30,461.09,SP,False,False
...,...,...,...,...,...,...,...,...,...
5518,3773,Cliente_3772,cliente3772@mail.com,5.511916e+12,2024-03-23,130.71,BA,False,False
5519,3795,Cliente_3794,cliente3794@mail.com,5.511932e+12,2024-01-28,414.1,BA,True,False
5520,2690,Cliente_2689,cliente2689@mail.com,5.511941e+12,2024-02-18,19.94,RJ,True,False
5521,1579,Cliente_1578,cliente1578@mail.com,5.511998e+12,2024-02-28,386.22,BA,True,False


,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado
0,1619,Cliente_1618,cliente1618@mail.com,5.511996e+12,2024-01-20,149.26,MG
1,4217,Cliente_4216,cliente4216@mail.com,5.511996e+12,2024-03-01,198.54,SP
2,1965,Cliente_1964,cliente1964@mail.com,5.511917e+12,2024-01-03,413.4,RS
3,3830,Cliente_3829,cliente3829@mail.com,5.511999e+12,2024-01-15,261.47,SP
4,2101,Cliente_2100,cliente2100@mail.com,5.511974e+12,2024-01-30,461.09,SP
...,...,...,...,...,...,...,...
5518,3773,Cliente_3772,cliente3772@mail.com,5.511916e+12,2024-03-23,130.71,BA
5519,3795,Cliente_3794,cliente3794@mail.com,5.511932e+12,2024-01-28,414.1,BA
5520,2690,Cliente_2689,cliente2689@mail.com,5.511941e+12,2024-02-18,19.94,RJ
5521,1579,Cliente_1578,cliente1578@mail.com,5.511998e+12,2024-02-28,386.22,BA


In [19]:
# Remove uma LINHA específica (por índice)
clientes3 = clientes2.drop(5522)
display(clientes3)


,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado
0,1619,Cliente_1618,cliente1618@mail.com,5.511996e+12,2024-01-20,149.26,MG
1,4217,Cliente_4216,cliente4216@mail.com,5.511996e+12,2024-03-01,198.54,SP
2,1965,Cliente_1964,cliente1964@mail.com,5.511917e+12,2024-01-03,413.4,RS
3,3830,Cliente_3829,cliente3829@mail.com,5.511999e+12,2024-01-15,261.47,SP
4,2101,Cliente_2100,cliente2100@mail.com,5.511974e+12,2024-01-30,461.09,SP
...,...,...,...,...,...,...,...
5517,3093,Cliente_3092,cliente3092@mail.com,5.511916e+12,2024-03-09,341.07,RJ
5518,3773,Cliente_3772,cliente3772@mail.com,5.511916e+12,2024-03-23,130.71,BA
5519,3795,Cliente_3794,cliente3794@mail.com,5.511932e+12,2024-01-28,414.1,BA
5520,2690,Cliente_2689,cliente2689@mail.com,5.511941e+12,2024-02-18,19.94,RJ


In [20]:
# Remove por LISTA de índices
clientes3 = clientes3.drop([10,80,90,100,200,300,400,500,600,700,800,900,1000])
display(clientes3)

,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado
0,1619,Cliente_1618,cliente1618@mail.com,5.511996e+12,2024-01-20,149.26,MG
1,4217,Cliente_4216,cliente4216@mail.com,5.511996e+12,2024-03-01,198.54,SP
2,1965,Cliente_1964,cliente1964@mail.com,5.511917e+12,2024-01-03,413.4,RS
3,3830,Cliente_3829,cliente3829@mail.com,5.511999e+12,2024-01-15,261.47,SP
4,2101,Cliente_2100,cliente2100@mail.com,5.511974e+12,2024-01-30,461.09,SP
...,...,...,...,...,...,...,...
5517,3093,Cliente_3092,cliente3092@mail.com,5.511916e+12,2024-03-09,341.07,RJ
5518,3773,Cliente_3772,cliente3772@mail.com,5.511916e+12,2024-03-23,130.71,BA
5519,3795,Cliente_3794,cliente3794@mail.com,5.511932e+12,2024-01-28,414.1,BA
5520,2690,Cliente_2689,cliente2689@mail.com,5.511941e+12,2024-02-18,19.94,RJ


In [21]:
clientes3.info()

<class 'pandas.DataFrame'>
Index: 5509 entries, 0 to 5521
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cliente_id   5509 non-null   int64  
 1   nome         5509 non-null   str    
 2   email        5336 non-null   str    
 3   telefone     4168 non-null   float64
 4   data_compra  5165 non-null   str    
 5   valor_gasto  5509 non-null   str    
 6   estado       5509 non-null   str    
dtypes: float64(1), int64(1), str(5)
memory usage: 611.2 KB


In [22]:
# Remove duplicatas considerando todas as colunas
clientes_final = clientes2.drop_duplicates(keep='first')
clientes_final.info()

<class 'pandas.DataFrame'>
Index: 5000 entries, 0 to 5522
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cliente_id   5000 non-null   int64  
 1   nome         5000 non-null   str    
 2   email        4850 non-null   str    
 3   telefone     3800 non-null   float64
 4   data_compra  4700 non-null   str    
 5   valor_gasto  5000 non-null   str    
 6   estado       5000 non-null   str    
dtypes: float64(1), int64(1), str(5)
memory usage: 554.9 KB


In [23]:
display(clientes_final)

,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado
0,1619,Cliente_1618,cliente1618@mail.com,5.511996e+12,2024-01-20,149.26,MG
1,4217,Cliente_4216,cliente4216@mail.com,5.511996e+12,2024-03-01,198.54,SP
2,1965,Cliente_1964,cliente1964@mail.com,5.511917e+12,2024-01-03,413.4,RS
3,3830,Cliente_3829,cliente3829@mail.com,5.511999e+12,2024-01-15,261.47,SP
4,2101,Cliente_2100,cliente2100@mail.com,5.511974e+12,2024-01-30,461.09,SP
...,...,...,...,...,...,...,...
5514,4427,Cliente_4426,cliente4426@mail.com,5.511951e+12,2024-01-03,258.44,RS
5516,467,Cliente_466,cliente466@mail.com,NaN,2024-02-06,R$ 413.07,BA
5517,3093,Cliente_3092,cliente3092@mail.com,5.511916e+12,2024-03-09,341.07,RJ
5518,3773,Cliente_3772,cliente3772@mail.com,5.511916e+12,2024-03-23,130.71,BA


#### 🎯 **Análise de Variáveis: `unique()`, `nunique()`, `value_counts()`**

Quando você tem uma **coluna categórica** (ex: estado, categoria de produto), você quer saber:
- Quais são os valores únicos?
- Quantas categorias diferentes existem?
- Qual categoria aparece mais vezes?

In [24]:

# unique() — lista todos os valores DIFERENTES
print(clientes_final['estado'].unique())
print('-'*20)

# nunique() — conta quantos valores diferentes existem
print(clientes_final['estado'].nunique())
print('-'*20)

# value_counts() — conta quantas vezes cada valor aparece (do maior pro menor)
print(clientes_final['estado'].value_counts())


<ArrowStringArray>
['MG', 'SP', 'RS', 'BA', 'RJ']
Length: 5, dtype: str
--------------------
5
--------------------
estado
BA    1055
RJ    1017
RS     987
MG     971
SP     970
Name: count, dtype: int64


In [25]:
# normalize=True — mostra em percentual ao invés de contagem
print(clientes_final['estado'].value_counts(normalize=True)*100)

estado
BA    21.10
RJ    20.34
RS    19.74
MG    19.42
SP    19.40
Name: proportion, dtype: float64


---

### 2️⃣ Remoção de Dados: `drop()`, `dropna()`, `drop_duplicates()`

Depois que você **identifica o problema**, você decide: o que fazer?

**3 opções principais:**
1. ❌ **Descartar a linha inteira** (se tiver muitos problemas)
2. ✅ **Preencher o buraco** (se for só um valor faltante — próximo tópico)
3. 🔄 **Substituir/corrigir** (se tiver um erro simples)

#### 🗑️ **Remover Linhas com `dropna()` e `drop_duplicates()`**

In [26]:
clientes_final.isna().sum()


cliente_id        0
nome              0
email           150
telefone       1200
data_compra     300
valor_gasto       0
estado            0
dtype: int64

In [27]:
clientes_final.isna().sum()

# dropna() — remove TODAS as linhas que têm QUALQUER valor faltante
# Cuidado: pode remover MUITAS linhas se tiver muitos NaN

cliente_final_limpo = clientes_final.dropna()




In [28]:
print(f"Antes: {len(clientes_final)} linhas | Depois: {len(cliente_final_limpo)} linhas")

Antes: 5000 linhas | Depois: 3800 linhas


In [29]:
cliente_final_limpo.isna().sum()

cliente_id     0
nome           0
email          0
telefone       0
data_compra    0
valor_gasto    0
estado         0
dtype: int64

In [30]:
# dropna() com subset — remove só se ESSA coluna específica tiver NaN
cliente_com_telefone = clientes_final.dropna(subset=['telefone'])
print(len(cliente_com_telefone))

# dropna() com how='all' — remove só se TODAS as colunas forem NaN (raro)
cliente_how_all = clientes_final.dropna(how='all')
print(len(cliente_how_all))


3800
5000


In [31]:
clientes_final_limpo.isna().sum()

NameError: name 'clientes_final_limpo' is not defined

In [ ]:
# Quais são essas linhas duplicadas?


In [ ]:
### JA FOI FALADO ANTERIORMENTE ###

 # drop_duplicates() com subset — considera duplicatas só dessa coluna


# keep='last' — ao invés de manter a primeira cópia, mantém a última


### 4️⃣ Substituição Condicional: `replace()` e `np.where()`

Às vezes você quer **encontrar um padrão errado e corrigir tudo de uma vez**.

**Analogia:** no estoque de um supermercado, você descobriu que todos os "banana" foram cadastrados como "banana prata" quando deveriam ser só "banana". Você quer uma operação que mude todos de uma vez.

#### 🔄 **Substituição Simples com `replace()`**

In [32]:
clientes_final['estado'].unique()

<ArrowStringArray>
['MG', 'SP', 'RS', 'BA', 'RJ']
Length: 5, dtype: str

In [ ]:
# replace() — substitui um valor por outro


In [36]:
# Substituir VÁRIOS valores de uma vez
clientes_final['estado'] = clientes_final['estado'].replace((
    'RJ' : 'Rio de Janeiro',
    'MG' : 'Minas Gerais',
    'SP' : 'São Paulo',
    'RS' : 'Rio Grande do Sul',
    'BA' : 'Bahia'
))

SyntaxError: invalid syntax (4251152073.py, line 3)

In [37]:
# replace() com regex (procura por padrão)

clientes_final['valor_gasto'] = clientes_final['valor_gasto'].replace(r'\s*R\$\s*', '', regex = True)

##### 🎯 **Substituição Condicional com `np.where()`**

Quando a lógica é mais complexa, `np.where()` é melhor.

**Analogia:** você quer classificar clientes por gasto: se gastar > 500, é "VIP", senão é "Normal".

In [42]:
clientes_final.info()

<class 'pandas.DataFrame'>
Index: 5000 entries, 0 to 5522
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cliente_id   5000 non-null   int64  
 1   nome         5000 non-null   str    
 2   email        4850 non-null   str    
 3   telefone     3800 non-null   float64
 4   data_compra  4700 non-null   str    
 5   valor_gasto  4999 non-null   float64
 6   estado       5000 non-null   str    
dtypes: float64(2), int64(1), str(4)
memory usage: 522.0 KB


In [39]:
# Alterando tipo de dado
clientes_final['valor_gasto'] = clientes_final['valor_gasto'].astype(float)

ValueError: could not convert string to float: 'None'

In [44]:
clientes_final['valor_gasto'] = pd.to_numeric(clientes_final['valor_gasto'], errors = 'coerce')

In [45]:
# np.where(condição, valor_se_verdade, valor_se_falso)
clientes_final['classe_clientes'] = np.where(
    clientes_final['valor_gasto'] > 200,
    'VIP',
    'Normal'
    )

display(clientes_final)

,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado,classe_clientes
0,1619,Cliente_1618,cliente1618@mail.com,5.511996e+12,2024-01-20,149.26,MG,Normal
1,4217,Cliente_4216,cliente4216@mail.com,5.511996e+12,2024-03-01,198.54,SP,Normal
2,1965,Cliente_1964,cliente1964@mail.com,5.511917e+12,2024-01-03,413.40,RS,VIP
3,3830,Cliente_3829,cliente3829@mail.com,5.511999e+12,2024-01-15,261.47,SP,VIP
4,2101,Cliente_2100,cliente2100@mail.com,5.511974e+12,2024-01-30,461.09,SP,VIP
...,...,...,...,...,...,...,...,...
5514,4427,Cliente_4426,cliente4426@mail.com,5.511951e+12,2024-01-03,258.44,RS,VIP
5516,467,Cliente_466,cliente466@mail.com,NaN,2024-02-06,413.07,BA,VIP
5517,3093,Cliente_3092,cliente3092@mail.com,5.511916e+12,2024-03-09,341.07,RJ,VIP
5518,3773,Cliente_3772,cliente3772@mail.com,5.511916e+12,2024-03-23,130.71,BA,Normal


In [46]:
# Você pode ANINHAR np.where() para mais categorias

clientes_final['classe_clientes'] = np.where(
    clientes_final['valor_gasto'] > 400,
    'VIP OURO',
    np.where(
        clientes_final['valor_gasto'] > 200,
        'VIP',
        'Normal'
    )
)
display(clientes_final)

,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado,classe_clientes
0,1619,Cliente_1618,cliente1618@mail.com,5.511996e+12,2024-01-20,149.26,MG,Normal
1,4217,Cliente_4216,cliente4216@mail.com,5.511996e+12,2024-03-01,198.54,SP,Normal
2,1965,Cliente_1964,cliente1964@mail.com,5.511917e+12,2024-01-03,413.40,RS,VIP OURO
3,3830,Cliente_3829,cliente3829@mail.com,5.511999e+12,2024-01-15,261.47,SP,VIP
4,2101,Cliente_2100,cliente2100@mail.com,5.511974e+12,2024-01-30,461.09,SP,VIP OURO
...,...,...,...,...,...,...,...,...
5514,4427,Cliente_4426,cliente4426@mail.com,5.511951e+12,2024-01-03,258.44,RS,VIP
5516,467,Cliente_466,cliente466@mail.com,NaN,2024-02-06,413.07,BA,VIP OURO
5517,3093,Cliente_3092,cliente3092@mail.com,5.511916e+12,2024-03-09,341.07,RJ,VIP
5518,3773,Cliente_3772,cliente3772@mail.com,5.511916e+12,2024-03-23,130.71,BA,Normal


In [47]:
clientes_final['classe_clientes'].unique()

<ArrowStringArray>
['Normal', 'VIP OURO', 'VIP']
Length: 3, dtype: str

In [49]:
# np.where() também funciona para corrigir erros
clientes_final['telefone'] = np.where(
    clientes_final['telefone'].isna(),
    'Telefone não informado',
    clientes_final['telefone']
)

display(clientes_final)

,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado,classe_clientes
0,1619,Cliente_1618,cliente1618@mail.com,5511995728617.0,2024-01-20,149.26,MG,Normal
1,4217,Cliente_4216,cliente4216@mail.com,5511995589119.0,2024-03-01,198.54,SP,Normal
2,1965,Cliente_1964,cliente1964@mail.com,5511917303817.0,2024-01-03,413.40,RS,VIP OURO
3,3830,Cliente_3829,cliente3829@mail.com,5511999387142.0,2024-01-15,261.47,SP,VIP
4,2101,Cliente_2100,cliente2100@mail.com,5511973959526.0,2024-01-30,461.09,SP,VIP OURO
...,...,...,...,...,...,...,...,...
5514,4427,Cliente_4426,cliente4426@mail.com,5511950654337.0,2024-01-03,258.44,RS,VIP
5516,467,Cliente_466,cliente466@mail.com,Telefone não informado,2024-02-06,413.07,BA,VIP OURO
5517,3093,Cliente_3092,cliente3092@mail.com,5511915949423.0,2024-03-09,341.07,RJ,VIP
5518,3773,Cliente_3772,cliente3772@mail.com,5511915686659.0,2024-03-23,130.71,BA,Normal


### 3️⃣ Tratamento de Valores Faltantes: `fillna()`

Às vezes remover não é a melhor opção. Você quer **preencher o buraco** com algo sensato.

**Analogia:** você tem um clínica com um formulário. Se um paciente deixa o campo "rua" vazio, você pode:
1. ❌ Jogar fora o paciente (extremo)
2. ✅ Pedir pra ele preencher depois (melhor)
3. ✅ Usar "Não informado" como padrão (razoável)
4. ✅ Usar a rua mais comum da sua base de dados (inteligente)

In [50]:
clientes_final.isna().sum()

cliente_id           0
nome                 0
email              150
telefone             0
data_compra        300
valor_gasto          1
estado               0
classe_clientes      0
dtype: int64

In [51]:
# fillna() com valor FIXO
clientes_final['email'] = clientes_final['email'].fillna('Email não informado')
clientes_final.isna().sum()



cliente_id           0
nome                 0
email                0
telefone             0
data_compra        300
valor_gasto          1
estado               0
classe_clientes      0
dtype: int64

In [55]:
# fillna() com MÉDIA (para números)
media_valor_gasto = clientes_final['valor_gasto'].mean().round(2)
print(media_valor_gasto)

clientes_final['valor_gasto'] = clientes_final['valor_gasto'].fillna(media_valor_gasto)

# fillna() com MEDIANA (melhor que média se houver atípicos)


251.02


In [56]:
clientes_final.isna().sum()

cliente_id           0
nome                 0
email                0
telefone             0
data_compra        300
valor_gasto          0
estado               0
classe_clientes      0
dtype: int64

In [63]:
# FFILL e BFILL
df = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 6],
    'categoria': ['A', np.nan, np.nan, 'B', np.nan, 'C']
})

print(df)

   id categoria
0   1         A
1   2       NaN
2   3       NaN
3   4         B
4   5       NaN
5   6         C


In [64]:
# fillna() com ffill (forward fill) — copia o valor ANTERIOR
'''df['categoria'] = df['categoria'].ffill()
print(df)'''

# fillna() com bfill (backward fill) — copia o valor POSTERIOR
df['categoria'] = df['categoria'].bfill()
print(df)

   id categoria
0   1         A
1   2         B
2   3         B
3   4         B
4   5         C
5   6         C


**Quando usar cada um?**
- **Valor fixo:** quando você quer marcar que é "faltante" (ex: "Não informado")
- **Média/mediana:** quando faz sentido numericamente (ex: salário, idade)
- **ffill/bfill:** quando os dados são **séries temporais** (ex: preço diário de uma ação)

---
### 5️⃣ Acesso e Modificação Precisa: `loc[]`, `iloc[]`, `at[]`, `iat[]`

Às vezes você quer **acessar ou modificar uma célula específica** (linha X, coluna Y).

**Analogia:** como encontrar a cadeira quebrada da 3ª fila, 5ª coluna do cinema.

In [78]:
# loc[] — acessa por RÓTULO (nome da coluna, índice da linha)
# iloc[] — acessa por POSIÇÃO numérica (linha 0, 1, 2...)

# Qual é o valor na linha de índice 100, coluna 'email'?
email = clientes_final.loc[100, 'email']
print(email)

# ou por posição: 100ª linha, 2ª coluna
email = clientes_final.iloc[100, 2]
print(email)

cliente422@mail.com
cliente422@mail.com


In [80]:
# Modificar uma célula
clientes_final.loc[100, 'email'] = "novoemail@mail.com"

email = clientes_final.loc[100, 'email']
print(email)

novoemail@mail.com


In [83]:
# Acessar MÚLTIPLAS linhas e colunas
# Todas as linhas 10 a 50, só as colunas 'nome' e 'email'
#display(clientes_final)

subset = clientes_final.loc[10:100, ['nome', 'email']]
display(subset)

,nome,email
10,Cliente_1467,cliente1467@mail.com
11,Cliente_3269,cliente3269@mail.com
12,Cliente_3154,cliente3154@mail.com
13,Cliente_4229,cliente4229@mail.com
14,Cliente_4140,cliente4140@mail.com
...,...,...
96,Cliente_1057,cliente1057@mail.com
97,Cliente_333,cliente333@mail.com
98,Cliente_4001,cliente4001@mail.com
99,Cliente_4812,cliente4812@mail.com


In [84]:
# Primeiras 10 linhas, primeiras 3 colunas
subset = clientes_final.iloc[:10, :3]
display(subset)

,cliente_id,nome,email
0,1619,Cliente_1618,cliente1618@mail.com
1,4217,Cliente_4216,cliente4216@mail.com
2,1965,Cliente_1964,cliente1964@mail.com
3,3830,Cliente_3829,cliente3829@mail.com
4,2101,Cliente_2100,cliente2100@mail.com
5,3429,Cliente_3428,cliente3428@mail.com
6,4583,Cliente_4582,cliente4582@mail.com
7,4017,Cliente_4016,cliente4016@mail.com
8,2463,Cliente_2462,cliente2462@mail.com
9,3434,Cliente_3433,cliente3433@mail.com


In [86]:
# Filtrar com CONDIÇÃO usando loc[]
# Todas as linhas onde valor_gasto > 400
clientes_vip = clientes_final.loc[clientes_final['valor_gasto']>400]
display(clientes_vip)

,cliente_id,nome,email,telefone,data_compra,valor_gasto,estado,classe_clientes
2,1965,Cliente_1964,cliente1964@mail.com,5511917303817.0,2024-01-03,413.40,RS,VIP OURO
4,2101,Cliente_2100,cliente2100@mail.com,5511973959526.0,2024-01-30,461.09,SP,VIP OURO
5,3429,Cliente_3428,cliente3428@mail.com,5511950414637.0,2024-02-17,443.56,BA,VIP OURO
7,4017,Cliente_4016,cliente4016@mail.com,5511923097108.0,2024-02-06,476.66,SP,VIP OURO
10,1468,Cliente_1467,cliente1467@mail.com,5511933641639.0,2024-03-14,417.26,RJ,VIP OURO
...,...,...,...,...,...,...,...,...
5477,2436,Cliente_2435,cliente2435@mail.com,5511994488390.0,2024-03-23,418.01,RS,VIP OURO
5492,976,Cliente_975,cliente975@mail.com,Telefone não informado,2024-01-27,479.79,RJ,VIP OURO
5513,3445,Cliente_3444,cliente3444@mail.com,5511972680520.0,2024-03-08,477.67,MG,VIP OURO
5516,467,Cliente_466,cliente466@mail.com,Telefone não informado,2024-02-06,413.07,BA,VIP OURO


In [89]:
# at[] e iat[] — acesso mais rápido a UMA célula (atalho)

print(clientes_final.at[100, 'email'])
print(clientes_final.iat[100, 2])

novoemail@mail.com
novoemail@mail.com


#### ✏️ **Exercícios de Fixação**

#### Exercício 1: Diagnóstico Rápido ⭐ 

**Enunciado:**
Você recebeu um arquivo CSV `alunos.csv` com dados de alunos. Sem abrir em Excel, use Python para:
1. Quantas linhas e colunas o arquivo tem?
2. Quais são os tipos de dados de cada coluna?
3. Qual coluna tem mais valores faltantes?
4. Quantas linhas duplicadas existem?

**Dica:** Use `shape`, `info()`, `isna().sum()`, `duplicated().sum()`




In [ ]:
alunos = pd.read_csv("C:/Temp/Exe/alunos.csv")

print("Exercicio 1")
print("Linhas, Colunas: ", alunos.shape, "\n")


Exercicio 1
Linhas, Colunas:  (11, 5) 



In [ ]:
print("Exercicio 2")
print(alunos.info(), "\n")


Exercicio 2
<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      11 non-null     int64  
 1   nome    11 non-null     str    
 2   idade   9 non-null      float64
 3   nota    10 non-null     float64
 4   email   8 non-null      str    
dtypes: float64(2), int64(1), str(2)
memory usage: 762.0 bytes
None 



In [ ]:
print("Exercicio 3")
print("Valores faltantes por coluna: \n", alunos.isna().sum(), "\n")


Exercicio 3
Valores faltantes por coluna: 
 id       0
nome     0
idade    2
nota     1
email    3
dtype: int64 



In [ ]:
print("Exercicio 4")
print("Linhas duplicadas: ", alunos.duplicated().sum())

Exercicio 4
Linhas duplicadas:  1


#### Exercício 2: Remover Valores Faltantes Inteligentemente ⭐

**Enunciado:**
O DataFrame `vendas` tem 10 linhas. Coluna 'email' tem 2 valores faltantes (5%), coluna 'telefone' tem 4 faltantes (40%), coluna 'valor' tem 2 faltantes (5%).

Para qual coluna você usa `dropna()` e para qual você usa `fillna()`? Por quê?

```python
# Seu código aqui:
# df = pd.read_csv('vendas.csv')
# ... completar
```

**Dica:** Pense em qual percentual é "aceitável" remover. 5% é OK, 40% é demais.


In [68]:
df = pd.read_csv("C:/Temp/vendas.csv", sep = ';')
df

,id,email,telefone,valor,valor_compra,estado
0,1,cliente1@email.com,2.199999e+10,100.0,100,SP
1,2,cliente2@email.com,NaN,200.0,200,RJ
2,3,NaN,2.199999e+10,150.0,150,MG
3,4,cliente4@email.com,NaN,NaN,300,SP
4,5,cliente5@email.com,2.199999e+10,250.0,250,RJ
5,6,cliente6@email.com,NaN,180.0,180,MG
6,7,cliente7@email.com,NaN,NaN,220,ES
7,8,cliente8@email.com,2.199999e+10,130.0,130,SP
8,9,NaN,2.199999e+10,170.0,170,RJ
9,10,cliente10@email.com,NaN,190.0,190,MG


In [74]:
df = df = pd.read_csv("C:/Temp/vendas.csv", sep = ';')
df

,id,email,telefone,valor,valor_compra,estado
0,1,cliente1@email.com,2.199999e+10,100.0,100,SP
1,2,cliente2@email.com,NaN,200.0,200,RJ
2,3,NaN,2.199999e+10,150.0,150,MG
3,4,cliente4@email.com,NaN,NaN,300,SP
4,5,cliente5@email.com,2.199999e+10,250.0,250,RJ
5,6,cliente6@email.com,NaN,180.0,180,MG
6,7,cliente7@email.com,NaN,NaN,220,ES
7,8,cliente8@email.com,2.199999e+10,130.0,130,SP
8,9,NaN,2.199999e+10,170.0,170,RJ
9,10,cliente10@email.com,NaN,190.0,190,MG


In [ ]:
df = pd.read_csv("C:/Temp/vendas.csv", sep = ';')

# Remove linhas onde email ou valor estão faltando ()
df = df.dropna(subset=["email", "valor"])

# Preenche telefones faltantes com um valor padrão
df["telefone"] = df["telefone"].fillna("Não informado")

display(df.reset_index())

,index,id,email,telefone,valor,valor_compra,estado
0,0,1,cliente1@email.com,21999990001.0,100.0,100,SP
1,1,2,cliente2@email.com,Não informado,200.0,200,RJ
2,4,5,cliente5@email.com,21999990005.0,250.0,250,RJ
3,5,6,cliente6@email.com,Não informado,180.0,180,MG
4,7,8,cliente8@email.com,21999990008.0,130.0,130,SP
5,9,10,cliente10@email.com,Não informado,190.0,190,MG


### Exercício 3: Limpeza Completa com Decisões ⭐⭐ 

**Enunciado:**
Você recebeu `produtos.csv`:

```
id,nome,preco,estoque,categoria,data_cadastro
1,Notebook,R$ 2500,150,Eletrônicos,2024-01-15
1,Notebook,R$ 2500,150,Eletrônicos,2024-01-15
2,Mouse,500,?,Periféricos,2024-01-16
3,Teclado,150.50,80,,2024-01-17
4,Monitor,,50,Eletrônicos,
```

Limpe esse dataset:
1. Remova duplicatas
2. Converta 'preco' de texto para número
3. Trate valores faltantes em 'estoque' (preencha com 0)
4. Trate valores faltantes em 'categoria' (preencha com 'Sem categoria')
5. Mostre as linhas onde 'data_cadastro' é vazia
6. Mostre quantas linhas sobraram

**Dica:** Use `drop_duplicates()`, `str.replace()`, `pd.to_numeric()`, `fillna()`


In [ ]:
import pandas as pd
import numpy as np

produtos = pd.Dataframe({
    'id' = [1, 1, 2, 3, 4],
    'nome' = ['notebook', 'notebook', 'mouse', 'teclado', 'monitor'],
    'preco' = 
})

In [ ]:
# 1. Remove duplicatas do DataFrame correto


In [ ]:
# 2. Converte preco de texto para número


In [ ]:
# 3. Trata estoque — preenche '?' e NaN com 0
#produtos['estoque'] = produtos['estoque'].replace('?', np.nan)  # transforma '?' em NaN


In [ ]:
# 4. Trata categoria — preenche vazio com 'Sem categoria'


In [ ]:
#caso precise fazer para mais de uma coluna



In [ ]:
# 5. Mostre as linhas onde data_cadastro é vazia


In [ ]:
# 6. Linhas resultantes do dataframe


### Exercício 4: Transformação com `np.where()` ⭐⭐ 

**Enunciado:**
Você tem `vendas.csv` com coluna 'valor_compra' (em números) e 'estado' (ex: SP, RJ, MG).

Crie uma nova coluna 'imposto':
- Se estado == 'SP': 18% de imposto
- Se estado == 'RJ': 20% de imposto
- Se estado == 'MG': 15% de imposto
- Outros: 17% de imposto

**Dica:** Use `np.where()`

### Exercício 5: Identificar e Corrigir Atípicos ⭐⭐ 

**Enunciado:**
Coluna 'idade' tem valores: `[25, 35, 1500, 28, 0, 42, -5, 50]` (há erros)

Corrija usando `np.where()`:
- Se idade < 18 ou > 100: marque como 'Inválido'
- Senão: mantenha o valor

Depois, remova as inválidas.

In [ ]:
df = pd.DataFrame({'idade': [25, 35, 1500, 28, 0, 42, -5, 50]})

